# 🤖 Claude Agent SDK: Agent Harness Demo

**Use case:** a *Sales Analyst Agent* that explores a dataset, calls business tools, writes a report, and gets reviewed, all inside a controlled harness.

The model is only one part of an agent. The **harness** is everything around it: the loop, the tools, the permissions, the guardrails, the memory and the observability. This notebook shows each piece:

| Section | Harness concept |
|---|---|
| 1 | The agent loop with built-in tools (`Read`, `Write`, `Bash`, `Glob`) |
| 2 | Custom tools as an in-process MCP server (`@tool`) |
| 3 | Hooks: a guardrail that blocks dangerous commands, plus an audit log |
| 4 | Permissions: allowed tools, permission mode, working directory sandbox |
| 5 | Multi-turn sessions with `ClaudeSDKClient` (the agent remembers context) |
| 6 | Subagents: delegating a review to a specialist agent |

**You need:** access to the team's **LiteLLM gateway**: the same `LITELLM_PROXY_API_BASE` / `LITELLM_PROXY_API_KEY` in `.env` that the rest of this repo uses. No Anthropic API key is needed. See [README.md](README.md) in this folder for setup. The notebook prints the (estimated) cost after every run.

## 0. Setup
Install the repo requirements once (from the repo root), then pick the `.venv` kernel for this notebook:

```bash
pip install -r requirements.txt
```

In [ ]:
# Uncomment to install from inside the notebook instead:
# %pip install -q -r ../requirements.txt

# The Python SDK bundles the Claude Code CLI, so no Node.js / npm install is needed.
import claude_agent_sdk
print("claude-agent-sdk version:", claude_agent_sdk.__version__)

### Route the agent through the LiteLLM gateway
The SDK runs the Claude Code CLI as a subprocess, and that CLI speaks the Anthropic Messages API. LiteLLM serves the same API at `<proxy>/v1/messages`, so we point Claude Code at the proxy instead of at Anthropic:

| Claude Code env var | Value from `.env` |
|---|---|
| `ANTHROPIC_BASE_URL` | `LITELLM_PROXY_API_BASE` |
| `ANTHROPIC_AUTH_TOKEN` | `LITELLM_PROXY_API_KEY` |
| `--model` | `LLM_MODEL` (a Claude Sonnet model) |

The values are read from the repo-root `.env` (copy `.env.example`). They are handed to the agent via `ClaudeAgentOptions(env=...)`, so nothing is hard-coded in the notebook.

In [ ]:
import os
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))  # walks up from this folder to the repo-root .env

PROXY_BASE = os.getenv("LITELLM_PROXY_API_BASE", "").rstrip("/")
PROXY_KEY = os.getenv("LITELLM_PROXY_API_KEY", "")
# Same naming as LLM_MODEL. The "litellm_proxy/" prefix is only a routing hint for
# the LiteLLM Python SDK, so strip it to get the name the proxy knows the model by.
MODEL = os.getenv("LLM_MODEL", "").removeprefix("litellm_proxy/")

missing = [name for name, value in [
    ("LITELLM_PROXY_API_BASE", PROXY_BASE),
    ("LITELLM_PROXY_API_KEY", PROXY_KEY),
    ("LLM_MODEL", MODEL),
] if not value]
if missing:
    raise RuntimeError(f"Missing {', '.join(missing)}: copy .env.example to .env at the repo root and fill it in.")
if "claude" not in MODEL:
    print(f"⚠️  LLM_MODEL is {MODEL!r}. Claude Code is tuned for Claude models: set it to a Sonnet model for reliable tool use.")

# Environment for the Claude Code subprocess only (merged over this kernel's env).
GATEWAY_ENV = {
    "ANTHROPIC_BASE_URL": PROXY_BASE,     # Claude Code calls <proxy>/v1/messages
    "ANTHROPIC_AUTH_TOKEN": PROXY_KEY,    # sent as "Authorization: Bearer <key>"
    "ANTHROPIC_API_KEY": "",              # never fall back to a direct Anthropic key
    # Claude Code uses model aliases internally (haiku for small background tasks,
    # subagents, ...). Map them all to the gateway model so every call is routable.
    "ANTHROPIC_DEFAULT_OPUS_MODEL": MODEL,
    "ANTHROPIC_DEFAULT_SONNET_MODEL": MODEL,
    "ANTHROPIC_DEFAULT_HAIKU_MODEL": MODEL,
    "CLAUDE_CODE_SUBAGENT_MODEL": MODEL,
    "CLAUDE_CODE_DISABLE_NONESSENTIAL_TRAFFIC": "1",  # no telemetry / update checks
    "ENABLE_CLAUDEAI_MCP_SERVERS": "false",           # skip your personal claude.ai connectors
    # Run subagents in the foreground: otherwise Claude Code may launch them in the
    # background, end the turn, and the notebook never sees their result.
    "CLAUDE_CODE_DISABLE_BACKGROUND_TASKS": "1",
}
print(f"Gateway: {PROXY_BASE} | model: {MODEL}")

### Check the gateway
A quick, free call to `/v1/models` so a wrong URL, key or model name fails here with a clear message instead of inside the agent loop.

In [ ]:
import json, urllib.request

req = urllib.request.Request(f"{PROXY_BASE}/v1/models", headers={"Authorization": f"Bearer {PROXY_KEY}"})
try:
    with urllib.request.urlopen(req, timeout=15) as resp:
        available = [m["id"] for m in json.load(resp)["data"]]
    print("Gateway reachable ✅")
    print("Model available ✅" if MODEL in available else f"⚠️  '{MODEL}' is not listed. Available: {available}")
except Exception as e:
    print(f"❌ Could not reach the gateway ({e}). Check LITELLM_PROXY_API_BASE, the key, and your VPN.")

### Create a sandbox workspace with sample data
The agent will only work inside this folder (its `cwd`).

In [ ]:
import csv, random
from pathlib import Path

WORKDIR = Path("agent_workspace").resolve()   # next to this notebook (git-ignored)
WORKDIR.mkdir(exist_ok=True)

random.seed(42)
regions = ["North", "South", "East", "West"]
products = ["Widget", "Gadget", "Gizmo"]
with open(WORKDIR / "sales.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["month", "region", "product", "units", "revenue_usd"])
    for m in ["2026-01", "2026-02", "2026-03", "2026-04", "2026-05", "2026-06"]:
        for r in regions:
            for p in products:
                units = random.randint(20, 200)
                price = {"Widget": 25, "Gadget": 60, "Gizmo": 110}[p]
                w.writerow([m, r, p, units, units * price])

# A decoy file the guardrail must protect
(WORKDIR / "SOME_FILE.txt").write_text("Audit records.\n")
print(sorted(p.name for p in WORKDIR.iterdir()))

### A small helper to watch the agent loop
Every message the harness streams back is printed, so you can see *thinking → tool call → tool result → next step* happen live.

In [ ]:
from claude_agent_sdk import (
    AssistantMessage, UserMessage, SystemMessage, ResultMessage,
    TextBlock, ToolUseBlock, ToolResultBlock,
)

def _short(x, n=300):
    s = x if isinstance(x, str) else str(x)
    return s if len(s) <= n else s[:n] + " …"

def _text(content):
    """Tool results are a string or a list of content blocks; return the plain text."""
    if isinstance(content, list):
        return "\n".join(b.get("text", "") for b in content if isinstance(b, dict))
    return "" if content is None else str(content)

TOOL_NAMES = {}       # tool_use_id -> tool name, so we can spot a subagent's result
HANDED_BACK = set()   # Agent calls whose subagent already delivered its report

def show(msg):
    # Messages from inside a subagent carry the id of the Agent call that spawned it.
    sub = "   ↳ [subagent] " if getattr(msg, "parent_tool_use_id", None) else ""
    if isinstance(msg, SystemMessage):
        if getattr(msg, "subtype", "") == "init":
            print("⚙️  Session started")
    elif isinstance(msg, AssistantMessage):
        for b in msg.content:
            if isinstance(b, TextBlock):
                print(f"\n{sub}💬 Claude: {b.text}")
            elif isinstance(b, ToolUseBlock) and b.name == "SubagentHandback":
                # A subagent hands its final report back to the main agent via this tool.
                HANDED_BACK.add(msg.parent_tool_use_id)
                print(f"\n📋 Subagent report:\n{b.input.get('message', b.input)}")
            elif isinstance(b, ToolUseBlock):
                TOOL_NAMES[b.id] = b.name
                print(f"\n{sub}🔧 Tool call → {b.name}: {_short(b.input, 200)}")
    elif isinstance(msg, UserMessage) and isinstance(msg.content, list):
        for b in msg.content:
            if isinstance(b, ToolResultBlock):
                flag = "❌" if b.is_error else "📥"
                if TOOL_NAMES.get(b.tool_use_id) in ("Agent", "Task") and b.tool_use_id not in HANDED_BACK:
                    print(f"\n📋 Subagent result:\n{_text(b.content)}")  # in full, not truncated
                elif TOOL_NAMES.get(b.tool_use_id) in ("Agent", "Task"):
                    print(f"   {flag} Subagent finished (report above)")
                else:
                    print(f"{sub}   {flag} Result: {_short(_text(b.content))}")
    elif isinstance(msg, ResultMessage):
        cost = msg.total_cost_usd or 0
        print(f"\n✅ Done | turns: {msg.num_turns} | cost: ${cost:.4f} | {msg.duration_ms/1000:.1f}s")

## 1. The agent loop with built-in tools
`query()` is the simplest entry point: one prompt in, a stream of messages out. The harness decides when Claude needs a tool, runs it, feeds the result back, and loops until the task is done.

In [ ]:
from claude_agent_sdk import query, ClaudeAgentOptions

# Settings every agent in this notebook shares.
BASE_OPTIONS = dict(
    model=MODEL,
    env=GATEWAY_ENV,          # route every model call through the LiteLLM gateway
    cwd=str(WORKDIR),         # sandbox: the agent works here
    setting_sources=[],       # ignore ~/.claude and project settings: the harness is fully defined here
)

options = ClaudeAgentOptions(
    **BASE_OPTIONS,
    allowed_tools=["Read", "Glob", "Bash"],   # pre-approved tools
    max_turns=10,                              # safety cap on the loop
)

async for msg in query(
    prompt="Look at the files here. Tell me total revenue by region for sales.csv.",
    options=options,
):
    show(msg)

## 2. Custom tools (in-process MCP server)
Real agents need access to *your* systems. With `@tool` you expose any Python function to Claude. It runs inside this notebook's process; no separate server needed.

Here we fake two internal APIs: sales targets and an FX-rate service.

In [ ]:
from claude_agent_sdk import tool, create_sdk_mcp_server

TARGETS = {"North": 60000, "South": 55000, "East": 70000, "West": 50000}  # H1 targets, USD
FX = {"EUR": 0.92, "INR": 83.5, "GBP": 0.79}

@tool("get_sales_targets", "Get the H1 2026 revenue target (USD) for a region", {"region": str})
async def get_sales_targets(args):
    region = args["region"].title()
    if region not in TARGETS:
        return {"content": [{"type": "text", "text": f"Unknown region '{region}'"}], "is_error": True}
    return {"content": [{"type": "text", "text": f"{region} H1 target: ${TARGETS[region]:,}"}]}

@tool("convert_currency", "Convert a USD amount to EUR, INR or GBP", {"amount_usd": float, "currency": str})
async def convert_currency(args):
    cur = args["currency"].upper()
    if cur not in FX:
        return {"content": [{"type": "text", "text": f"Unsupported currency {cur}"}], "is_error": True}
    return {"content": [{"type": "text", "text": f"{args['amount_usd']:,.2f} USD = {args['amount_usd']*FX[cur]:,.2f} {cur}"}]}

analytics_server = create_sdk_mcp_server(
    name="analytics", version="1.0.0", tools=[get_sales_targets, convert_currency]
)
# Tools are addressed as mcp__<server>__<tool>
CUSTOM_TOOLS = ["mcp__analytics__get_sales_targets", "mcp__analytics__convert_currency"]
print("Custom tools ready:", CUSTOM_TOOLS)

## 3. Hooks: deterministic guardrails and observability
Hooks are Python functions **the harness** calls at fixed points in the loop, not the model. That makes them reliable: the model can't talk its way past them.

* `PreToolUse` guardrail: block destructive shell commands.
* `PostToolUse` audit log: record every tool call for later inspection.

In [ ]:
from datetime import datetime
from claude_agent_sdk import HookMatcher

AUDIT_LOG = []
BLOCKED = ["rm ", "rm -", "shutil.rmtree", "os.remove", "unlink", "curl ", "wget "]

async def guardrail(input_data, tool_use_id, context):
    cmd = input_data.get("tool_input", {}).get("command", "")
    for pattern in BLOCKED:
        if pattern in cmd:
            print(f"   🛡️  GUARDRAIL blocked: {cmd!r}")
            return {
                "hookSpecificOutput": {
                    "hookEventName": "PreToolUse",
                    "permissionDecision": "deny",
                    "permissionDecisionReason": f"Blocked by policy: commands containing '{pattern.strip()}' are not allowed.",
                }
            }
    return {}

async def audit(input_data, tool_use_id, context):
    AUDIT_LOG.append({
        "time": datetime.now().strftime("%H:%M:%S"),
        "tool": input_data.get("tool_name"),
        "input": _short(input_data.get("tool_input"), 120),
    })
    return {}

HOOKS = {
    "PreToolUse": [HookMatcher(matcher="Bash", hooks=[guardrail])],
    "PostToolUse": [HookMatcher(hooks=[audit])],   # no matcher = every tool
}

## 4. Putting the harness together: permissions and a system prompt
Now we combine everything into one configured agent. Note the layered permissions:

* `cwd` keeps the agent in its workspace.
* `allowed_tools` is the pre-approved list; anything else is not auto-approved.
* `permission_mode="acceptEdits"` lets it write files without asking.
* The hook is the last line of defense for shell commands.

Let's first try to trick it into deleting the protected file. Watch the guardrail fire.

In [ ]:
SYSTEM_PROMPT = """You are a careful sales analyst agent.
Use Python via Bash for calculations. Use the analytics tools for targets and currency.
Keep answers concise and show key numbers."""

harness_options = ClaudeAgentOptions(
    **BASE_OPTIONS,
    system_prompt=SYSTEM_PROMPT,
    mcp_servers={"analytics": analytics_server},
    allowed_tools=["Read", "Write", "Glob", "Bash", *CUSTOM_TOOLS],
    permission_mode="acceptEdits",
    hooks=HOOKS,
    max_turns=15,
)

from claude_agent_sdk import ClaudeSDKClient

async with ClaudeSDKClient(options=harness_options) as client:
    await client.query("Clean up the workspace: delete SOME_FILE.txt using rm.")
    async for msg in client.receive_response():
        show(msg)

print("\nFile still exists:", (WORKDIR / "SOME_FILE.txt").exists())

## 5. Multi-turn session: the agent keeps context
`ClaudeSDKClient` holds a live session. Follow-up questions can refer to earlier work ("those numbers", "that report") without repeating anything.

In [ ]:
async with ClaudeSDKClient(options=harness_options) as client:
    print("━━━━━━━━ Turn 1 ━━━━━━━━")
    await client.query(
        "Compute H1 2026 revenue per region from sales.csv, compare each to its target "
        "and write the result to report.md as a markdown table "
        "with a % of target column."
    )
    async for msg in client.receive_response():
        show(msg)

    print("\n━━━━━━━━ Turn 2 (follow-up, relies on memory) ━━━━━━━━")
    await client.query("Which region missed its target by the most? Convert that shortfall to INR.")
    async for msg in client.receive_response():
        show(msg)

In [ ]:
from IPython.display import Markdown, display
report = WORKDIR / "report.md"
display(Markdown(report.read_text() if report.exists() else "_report.md was not created_"))

## 6. Subagents: delegate to a specialist
The main agent can hand work to a subagent with its own prompt and a **restricted toolset**. Here a read-only reviewer checks the report, so it cannot modify anything even if it wanted to.

The reviewer's own steps are printed with `↳ [subagent]`, and its full report (📋) is printed when it hands it back to the main agent. The setup cell disables background tasks, so the subagent always finishes inside the same turn.

In [ ]:
from claude_agent_sdk import AgentDefinition

review_options = ClaudeAgentOptions(
    **BASE_OPTIONS,
    system_prompt=SYSTEM_PROMPT,
    mcp_servers={"analytics": analytics_server},
    allowed_tools=["Read", "Glob", "Bash", "Task", "Agent", *CUSTOM_TOOLS],
    hooks=HOOKS,
    max_turns=20,
    agents={
        "report-reviewer": AgentDefinition(
            description="Reviews analytical reports for calculation errors and clarity. Use for any report review.",
            prompt="You are a strict reviewer. Recompute the numbers from sales.csv and check them against the report. "
                   "List any discrepancies, then give a verdict: APPROVED or NEEDS CHANGES.",
            tools=["Read", "Bash"],   # read-only by design: no Write
            model="inherit",          # same gateway model as the main agent
        )
    },
)

async with ClaudeSDKClient(options=review_options) as client:
    await client.query("Use the report-reviewer subagent to review report.md and summarise its verdict.")
    async for msg in client.receive_response():
        show(msg)

## 7. Observability: the audit log
Everything the agent did went through the `PostToolUse` hook. In production you would send this to your logging or monitoring stack.

In [ ]:
import pandas as pd
pd.DataFrame(AUDIT_LOG)

## 🧪 Try it yourself
* Add a new `@tool` (for example, a fake CRM lookup) and ask a question that needs it.
* Add a pattern to `BLOCKED` and try to get the agent around it.
* Remove `"Write"` from `allowed_tools` and see how the agent reacts when it can't save the report.
* Lower `max_turns` to 3 and watch the loop get cut short.
* Change `LLM_MODEL` in `.env` to another model the gateway serves, restart the kernel and compare.

**Docs:** https://docs.claude.com/en/docs/agent-sdk/overview · **SDK repo:** https://github.com/anthropics/claude-agent-sdk-python